# Reviewer Revision — Stage A2: Canonical MNB Reconciliation

Stage A revealed an important inconsistency:

- The **reproduced leakage-clean Phase-2 MNB** gives approximately Accuracy **0.7399**, Macro-F1 **0.7360**.
- The persistent Phase-5 comparator gives approximately Accuracy **0.7441**, Macro-F1 **0.7401**.

Before manuscript revision, this notebook makes the comparison internally consistent by:

1. reading the exact Stage-A selected MNB parameters;
2. rebuilding the leakage-clean source training set;
3. generating **canonical MNB predictions on all five evaluation sets**;
4. comparing them with the old persistent Phase-5 MNB predictions;
5. recomputing all paired bootstrap and McNemar tests against the existing three RoBERTa seed predictions;
6. applying Holm-Bonferroni correction.

The output of this notebook should become the canonical NB-vs-RoBERTa statistical comparison in the revised paper.

In [ ]:
!pip -q install datasets openpyxl scipy statsmodels

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re, hashlib, json, warnings, requests
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score,
    recall_score, precision_score, confusion_matrix
)
from scipy.stats import chi2, binomtest
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

REV = Path("/content/drive/MyDrive/stress_detection_reviewer_revision")
PERSIST = Path("/content/drive/MyDrive/stress_detection_phase5_persistent")
PERSIST_PRED = PERSIST/"predictions"

OUT = REV/"stageA2"
OUT.mkdir(parents=True, exist_ok=True)

print("Revision folder:", REV)
print("Persistent Phase-5 prediction folder exists:", PERSIST_PRED.exists())

In [ ]:
# Read exact selected parameters from Stage A.
audit = pd.read_csv(REV/"mnb_protocol_audit.csv")
row = audit[audit["protocol"]=="leakage_clean"].iloc[0]
params = json.loads(row["best_params"])

alpha = float(params["clf__alpha"])
max_features = int(params["tfidf__max_features"])

print("Canonical MNB parameters from Stage A:")
print("alpha =", alpha)
print("max_features =", max_features)
print("Stage-A CV Macro-F1 =", row["best_cv_macro_f1"])

In [ ]:
def normalize_text_for_audit(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"https?://\S+|www\.\S+", "<url>", text)
    text = re.sub(r"\s+", " ", text)
    return text

def text_hash(text):
    return hashlib.sha256(normalize_text_for_audit(text).encode("utf-8")).hexdigest()

ds = load_dataset("andreagasparini/dreaddit")
train_raw = ds["train"].to_pandas()[["text","label"]].copy()
test = ds["test"].to_pandas()[["text","label"]].copy()

train_raw["text_hash"] = train_raw["text"].map(text_hash)
test["text_hash"] = test["text"].map(text_hash)

overlap = set(train_raw["text_hash"]) & set(test["text_hash"])
clean = train_raw.drop_duplicates("text_hash", keep="first")
clean = clean[~clean["text_hash"].isin(overlap)].reset_index(drop=True)

print("Leakage-clean train:", len(clean))
print("Test:", len(test))

In [ ]:
# Rebuild external primary sets exactly as in Stage A.
RAW = Path("/content/reviewer_revision_stageA2_raw")
RAW.mkdir(exist_ok=True)

FILES = {
    "Reddit_Title":"Reddit_Title.xlsx",
    "Reddit_Combi":"Reddit_Combi.xlsx",
    "Twitter_Full":"Twitter_Full.xlsx",
    "Twitter_NonAdvert":"Twitter_Non-Advert.xlsx",
}
BASE = "https://raw.githubusercontent.com/SenticNet/stress-detection/master"

def prep(df):
    df = df.copy()
    df["text"] = df["text"].astype("string").str.strip()
    df.loc[df["text"]=="","text"] = pd.NA
    df["label"] = pd.to_numeric(df["label"],errors="coerce")
    df = df.dropna(subset=["text","label"]).copy()
    df["label"] = df["label"].astype(int)
    df["text_hash"] = df["text"].map(text_hash)
    return df

external = {}
for name, fn in FILES.items():
    p = RAW/fn
    if not p.exists():
        rr = requests.get(f"{BASE}/{fn}", timeout=60)
        rr.raise_for_status()
        p.write_bytes(rr.content)
    raw = pd.read_excel(p)

    if name=="Reddit_Title":
        text, label = raw["title"], raw["label"]
    elif name=="Reddit_Combi":
        text = (raw["title"].fillna("").astype(str).str.strip() + " " +
                raw["body"].fillna("").astype(str).str.strip()).str.strip()
        label = raw["label"]
    elif name=="Twitter_Full":
        text, label = raw["text"], raw["labels"]
    else:
        text, label = raw["text"], raw["label"]

    d = prep(pd.DataFrame({"text":text,"label":label}))
    nuniq = d.groupby("text_hash")["label"].nunique()
    conflicts = set(nuniq[nuniq>1].index)
    d = d[~d["text_hash"].isin(conflicts)].drop_duplicates("text_hash", keep="first").reset_index(drop=True)
    external[name] = d

print({k: len(v) for k,v in external.items()})

## Fit the canonical MNB and save predictions for all five evaluation sets

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.98,
    max_features=max_features
)
Xtr = tfidf.fit_transform(clean["text"].astype(str))
mnb = MultinomialNB(alpha=alpha).fit(Xtr, clean["label"].astype(int))

EVAL = {"Dreaddit_Test": test.reset_index(drop=True), **external}

def metric_row(y, pred, p_pos):
    tn, fp, fn, tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(
        accuracy=accuracy_score(y,pred),
        macro_f1=f1_score(y,pred,average="macro",zero_division=0),
        balanced_accuracy=balanced_accuracy_score(y,pred),
        macro_precision=precision_score(y,pred,average="macro",zero_division=0),
        macro_recall=recall_score(y,pred,average="macro",zero_division=0),
        positive_recall=recall_score(y,pred,pos_label=1,zero_division=0),
        roc_auc=roc_auc_score(y,p_pos),
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp)
    )

metric_rows=[]
for name, df in EVAL.items():
    Xt = tfidf.transform(df["text"].astype(str))
    pred = mnb.predict(Xt)
    p_pos = mnb.predict_proba(Xt)[:,1]
    y = df["label"].astype(int).to_numpy()

    out = pd.DataFrame({
        "text":df["text"].astype(str).reset_index(drop=True),
        "true_label":y,
        "predicted_label":pred,
        "p_pos":p_pos
    })
    out.to_csv(OUT/f"MNB_CANONICAL_{name}.csv", index=False)
    metric_rows.append({"dataset":name, **metric_row(y,pred,p_pos)})

canonical_metrics = pd.DataFrame(metric_rows)
display(canonical_metrics.round(6))
canonical_metrics.to_csv(OUT/"canonical_mnb_metrics.csv", index=False)

## Compare canonical MNB with the old persistent Phase-5 MNB

This quantifies exactly how much the old comparator differed.

In [ ]:
compare_rows=[]
for name in EVAL:
    new = pd.read_csv(OUT/f"MNB_CANONICAL_{name}.csv")
    oldp = PERSIST_PRED/f"NB_{name}.csv"
    if not oldp.exists():
        continue
    old = pd.read_csv(oldp)

    assert len(new)==len(old), (name, len(new), len(old))
    assert np.array_equal(new["true_label"].to_numpy(), old["true_label"].to_numpy()), name

    disagreements = int((new["predicted_label"].to_numpy()!=old["predicted_label"].to_numpy()).sum())
    compare_rows.append({
        "dataset":name,
        "n":len(new),
        "prediction_disagreements":disagreements,
        "disagreement_rate":disagreements/len(new),
        "canonical_macro_f1":f1_score(new.true_label,new.predicted_label,average="macro"),
        "old_phase5_macro_f1":f1_score(old.true_label,old.predicted_label,average="macro")
    })

comparison = pd.DataFrame(compare_rows)
display(comparison.round(6))
comparison.to_csv(OUT/"canonical_vs_old_phase5_mnb.csv", index=False)

## Recompute paired bootstrap and McNemar using canonical MNB

In [ ]:
def paired_bootstrap(y, pred_nb, pred_rb, B=2000, seed=2026):
    rng = np.random.default_rng(seed)
    n = len(y)
    obs = (
        f1_score(y,pred_rb,average="macro",zero_division=0) -
        f1_score(y,pred_nb,average="macro",zero_division=0)
    )
    vals = np.empty(B)
    for b in range(B):
        idx = rng.integers(0,n,size=n)
        vals[b] = (
            f1_score(y[idx],pred_rb[idx],average="macro",zero_division=0) -
            f1_score(y[idx],pred_nb[idx],average="macro",zero_division=0)
        )
    lo, hi = np.quantile(vals,[0.025,0.975])
    # descriptive two-sided sign-based bootstrap p approximation
    pboot = 2*min(np.mean(vals<=0), np.mean(vals>=0))
    return obs, lo, hi, min(float(pboot),1.0)

def mcnemar_test(y, pred_nb, pred_rb):
    nb_ok = pred_nb==y
    rb_ok = pred_rb==y
    b = int(np.sum(nb_ok & ~rb_ok))   # NB correct, RB wrong
    c = int(np.sum(~nb_ok & rb_ok))   # NB wrong, RB correct
    discordant = b+c
    if discordant < 25:
        p = binomtest(min(b,c), n=discordant, p=0.5, alternative="two-sided").pvalue if discordant else 1.0
        stat = np.nan
        method = "exact_binomial"
    else:
        stat = (abs(b-c)-1)**2/discordant
        p = chi2.sf(stat,1)
        method = "continuity_corrected_chi2"
    return b,c,discordant,stat,p,method

boot_rows=[]
mc_rows=[]

for name in EVAL:
    nbp = pd.read_csv(OUT/f"MNB_CANONICAL_{name}.csv")
    y = nbp.true_label.to_numpy().astype(int)
    pred_nb = nbp.predicted_label.to_numpy().astype(int)

    for seed in [42,52,62]:
        rbp = PERSIST_PRED/f"RB_{seed}_{name}.csv"
        if not rbp.exists():
            raise FileNotFoundError(f"Missing {rbp}")
        rb = pd.read_csv(rbp)
        pred_rb = rb.predicted_label.to_numpy().astype(int)

        obs,lo,hi,pb = paired_bootstrap(y,pred_nb,pred_rb,B=2000,seed=1000+seed)
        boot_rows.append({
            "dataset":name,"seed":seed,
            "f1_diff_roberta_minus_nb":obs,
            "ci_low":lo,"ci_high":hi,
            "bootstrap_sign_p":pb
        })

        b,c,d,stat,p,method = mcnemar_test(y,pred_nb,pred_rb)
        mc_rows.append({
            "dataset":name,"seed":seed,
            "nb_correct_rb_wrong":b,
            "nb_wrong_rb_correct":c,
            "discordant":d,
            "statistic":stat,
            "p_value":p,
            "method":method
        })

boot = pd.DataFrame(boot_rows)
mc = pd.DataFrame(mc_rows)

reject,padj,_,_ = multipletests(mc["p_value"].values, alpha=.05, method="holm")
mc["holm_adjusted_p"] = padj
mc["holm_significant_0_05"] = reject

display(boot.round(6))
display(mc.round(8))

boot.to_csv(OUT/"paired_bootstrap_CANONICAL_MNB.csv", index=False)
mc.to_csv(OUT/"mcnemar_CANONICAL_MNB_holm.csv", index=False)

## Compact manuscript-ready summary

In [ ]:
seed_summary = (boot.groupby("dataset")
                .agg(mean_f1_difference=("f1_diff_roberta_minus_nb","mean"),
                     min_f1_difference=("f1_diff_roberta_minus_nb","min"),
                     max_f1_difference=("f1_diff_roberta_minus_nb","max"))
                .reset_index())

seed_summary["all_bootstrap_CIs_exclude_zero"] = seed_summary["dataset"].map(
    boot.groupby("dataset").apply(
        lambda d: bool((((d.ci_low>0)&(d.ci_high>0)) | ((d.ci_low<0)&(d.ci_high<0))).all())
    )
)
seed_summary["all_mcnemar_holm_significant"] = seed_summary["dataset"].map(
    mc.groupby("dataset")["holm_significant_0_05"].all()
)

display(seed_summary.round(6))
seed_summary.to_csv(OUT/"CANONICAL_STATISTICAL_SUMMARY.csv", index=False)

print("Largest Holm-adjusted p:", mc.holm_adjusted_p.max())
print("\nSaved all canonical reconciliation outputs to:")
print(OUT)

## Send back

Please send screenshots of:

1. `canonical_mnb_metrics`
2. `canonical_vs_old_phase5_mnb`
3. `paired_bootstrap_CANONICAL_MNB`
4. `mcnemar_CANONICAL_MNB_holm`
5. `CANONICAL_STATISTICAL_SUMMARY`

After this reconciliation is complete, we can safely update the manuscript's MNB values and statistical claims, then proceed to Stage B (all-six-model transfer + diagnostic analysis).